In [ ]:
!pip install yfinance
!pip install chronos-forecasting
!pip install transformers==4.40.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

### ‘I acknowledge the use of ChatGPT & Grok to plan my essay, generate some ideas for the content, and implement the codes with our supervision‘

### Preprocessing the Pretrained Results for the Predictions

In [ ]:
import yfinance as yf
import numpy as np
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM
from chronos import BaseChronosPipeline
import torch
import pandas as pd
from sklearn.metrics import r2_score
import os

# Define stocks and parameters
stocks = [
    "PLTR", "TSLA", "KKR", "AMD", "NVDA", "BX", "BA", "AMAT", "LRCX", "C",
    "DIS", "BKNG", "ISRG", "GS", "UBER", "MS", "BAC", "ADBE", "CRM", "BLK",
    "NFLX", "KLAC", "QCOM", "INTU", "AXP", "ACN", "GE", "SPGI", "AAPL", "META",
    "COP", "MU", "WFC", "CRWD", "AMZN", "PANW", "PLD", "JPM", "CAT", "LOW",
    "MA", "CVX", "ANET", "INTC", "UNP", "HD", "ORCL", "HON", "ETN", "ADI"
]

sequence_sizes = [20, 126, 252]  # Different sequence lengths
forecast_horizons = [1, 5, 20, 126, 252]  # Forecast horizons: 1 day, ~week, ~month, ~half-year, ~year
max_horizon = max(forecast_horizons)
batch_size = 32  # Batch size for processing
output_dir = "/content/drive/MyDrive/financial_data/detect_easy"

# Define multiple models and sequence sizes to loop through
models = [
    {"name": "Maple728/TimeMoE-50M", "type": "time_moe"},
    {"name": "Maple728/TimeMoE-200M", "type": "time_moe"},
    {"name": "amazon/chronos-bolt-small", "type": "chronos"},
    {"name": "amazon/chronos-bolt-base", "type": "chronos"}
]

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Check which stocks have already been processed
processed_stocks = {f.split('_')[0] for f in os.listdir(output_dir) if f.endswith('_metrics.csv')}
stocks_to_process = [stock for stock in stocks if stock not in processed_stocks]

print(f"Processing {len(stocks_to_process)} unprocessed stocks out of {len(stocks)} total.")

for stock in stocks_to_process:
    # Download stock data
    data = yf.download(stock, start='2008-01-01', end='2021-01-01')
    closing_prices = data['Close'].values.reshape(-1, 1)
    dates = data.index

    # Initialize list to collect all metrics for this stock
    stock_data = []

    # Loop through each model
    for model_config in models:
        model_name = model_config["name"]
        model_type = model_config["type"]

        # Load the model based on type
        if model_type == "time_moe":
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="cuda" if torch.cuda.is_available() else "cpu",
                trust_remote_code=True
            )
            model.to(device)
            model.eval()
        elif model_type == "chronos":
            model = BaseChronosPipeline.from_pretrained(
                model_name,
                device_map="cuda" if torch.cuda.is_available() else "cpu",
                torch_dtype=torch.bfloat16
            )

        # Loop through each sequence size
        for seq_size in sequence_sizes:
            context_length = seq_size

            # Create sequences and targets based on current sequence size
            sequences = []
            targets = []
            sequence_indices = list(range(context_length, len(closing_prices) - max_horizon))

            for i in sequence_indices:
                sequences.append(closing_prices[i - context_length:i])
                targets.append(closing_prices[i:i + max_horizon])

            sequences = np.array(sequences)
            targets = np.array(targets)

            # Process in batches with batch-wise normalization
            scaler = StandardScaler()

            with torch.no_grad():
                for i in range(0, len(sequences), batch_size):
                    batch_size_curr = min(batch_size, len(sequences) - i)
                    batch_sequences = sequences[i:i + batch_size_curr]
                    batch_targets = targets[i:i + batch_size_curr]
                    batch_indices = sequence_indices[i:i + batch_size_curr]

                    # Normalize within the batch
                    batch_sequences_flat = batch_sequences.reshape(-1, 1)
                    batch_sequences_scaled = scaler.fit_transform(batch_sequences_flat).reshape(batch_sequences.shape)
                    batch_targets_scaled = scaler.transform(batch_targets.reshape(-1, 1)).reshape(batch_targets.shape)

                    # Prepare input tensor
                    past_values = torch.tensor(batch_sequences_scaled, dtype=torch.float32).squeeze(-1).to(device)

                    # Generate predictions based on model type
                    if model_type == "time_moe":
                        outputs = model.generate(inputs=past_values, max_new_tokens=252)
                        predicted_values = outputs[:, context_length:context_length + max_horizon].cpu().numpy()
                    elif model_type == "chronos":
                        # Chronos expects 2D tensor [batch, sequence_length]
                        _, predicted_values = model.predict_quantiles(
                            context=past_values,
                            prediction_length=max_horizon,
                            quantile_levels=[0.5]  # Use median for consistency
                        )
                        predicted_values = predicted_values.squeeze(-1).cpu().numpy()  # Shape: [batch, prediction_length]

                    # Calculate metrics for each horizon
                    for h in forecast_horizons:
                        predicted = predicted_values[:, h - 1]
                        actual = batch_targets_scaled[:, h - 1, 0]

                        # MSE
                        mse = (predicted - actual) ** 2
                        # MAE
                        mae = np.abs(predicted - actual)
                        # MAPE (handle division by zero)
                        mape = np.abs((actual - predicted) / (actual + 1e-10)) * 100
                        # R² (batch-level)
                        r2 = r2_score(actual, predicted) if len(actual) > 1 else 0
                        # Direction correctness
                        pred_direction = np.sign(predicted - batch_sequences_scaled[:, -1, 0])
                        actual_direction = np.sign(actual - batch_sequences_scaled[:, -1, 0])
                        direction_correct = (pred_direction == actual_direction).astype(int)

                        # Record data with model and sequence size
                        for j in range(len(batch_sequences)):
                            pred_date = dates[batch_indices[j] + h - 1]
                            stock_data.append({
                                'stock': stock,
                                'model': model_name,
                                'sequence_size': seq_size,
                                'date': pred_date,
                                'horizon': h,
                                'mse': mse[j],
                                'mae': mae[j],
                                'mape': mape[j],
                                'r2': r2,
                                'direction_correct': direction_correct[j]
                            })

    # Create dataframe for this stock with all collected data
    df_stock = pd.DataFrame(stock_data)

    # Save to CSV
    output_file = os.path.join(output_dir, f"{stock}_metrics_v2.csv")
    df_stock.to_csv(output_file, index=False)
    print(f"Saved metrics for {stock} to {output_file}")

print("Processing complete.")

Processing 50 unprocessed stocks out of 50 total.
YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

configuration_time_moe.py:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Maple728/TimeMoE-50M:
- configuration_time_moe.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/chronos/chronos_bolt.py:527: UserWarning: We recommend keeping prediction length <= 64. The quality of longer 

Saved metrics for META to /content/drive/MyDrive/financial_data/detect_easy/META_metrics_v2.csv


[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/chronos/chronos_bolt.py:527: UserWarning: 

Saved metrics for COP to /content/drive/MyDrive/financial_data/detect_easy/COP_metrics_v2.csv


[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



### Generating the Finbert Embedding

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModel, AutoTokenizer

# Load FinBERT model and tokenizer
finbert_model = AutoModel.from_pretrained('ProsusAI/finbert').to("cuda" if torch.cuda.is_available() else "cpu")
finbert_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

# Define output directory and create it if it doesn't exist
output_dir = "/content/drive/MyDrive/financial_data/finbert_emb_v2"
os.makedirs(output_dir, exist_ok=True)

def get_sentiment_features(headlines, finbert_model, finbert_tokenizer):
    """
    Process a list of headlines through FinBERT in batch.
    Returns a tensor of shape (n_headlines, 768) of raw embeddings.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not headlines:
        return torch.empty(0, 768)  # Handle empty list case

    # Tokenize all headlines at once
    inputs = finbert_tokenizer(
        headlines,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    # Compute embeddings without gradient computation
    with torch.no_grad():
        outputs = finbert_model(**inputs)

    # Average across tokens for each headline: (n_headlines, 768)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu()

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/financial_data/merged_dataset.csv")

# Set stock_name to 'general' where is_general = 1
df.loc[df['is_general'] == 1, 'stock_name'] = 'general'

stocks = [
    "PLTR", "TSLA", "KKR", "AMD", "NVDA", "BX", "BA", "AMAT", "LRCX", "C",
    "DIS", "BKNG", "ISRG", "GS", "UBER", "MS", "BAC", "ADBE", "CRM", "BLK",
    "NFLX", "KLAC", "QCOM", "INTU", "AXP", "ACN", "GE", "SPGI", "AAPL", "META",
    "COP", "MU", "WFC", "CRWD", "AMZN", "PANW", "PLD", "JPM", "CAT", "LOW",
    "MA", "CVX", "ANET", "INTC", "UNP", "HD", "ORCL", "HON", "ETN", "ADI", "general"
]

# Select the stock names
df = df[df['stock_name'].isin(stocks)]

# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'])

# DropNA value
df = df.dropna()

# Group by 'Date' and 'stock_name', aggregating headlines into lists
grouped = df.groupby(['Date', 'stock_name']).agg({'Headlines': list})

<ipython-input-40-dbe04f8ee079>:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'])


In [ ]:
from tqdm import tqdm
import os
import torch
import pandas as pd

# Assuming grouped is your DataFrameGroupBy object or similar iterable

file_paths = []


def get_sentiment_features(headlines, finbert_model, finbert_tokenizer):
    """
    Process a list of headlines through FinBERT in batch.
    Returns a tensor of shape (n_headlines, 768) of raw embeddings.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not headlines:
        return torch.empty(0, 768)  # Handle empty list case

    # Tokenize all headlines at once
    inputs = finbert_tokenizer(
        headlines,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    # Compute embeddings without gradient computation
    with torch.no_grad():
        outputs = finbert_model(**inputs)

    # Average across tokens for each headline: (n_headlines, 768)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu()

for (date, stock_name), row in tqdm(grouped.iterrows(), total=len(grouped), desc="Processing embeddings"):
    headlines = row['Headlines']  # List of headlines for this date and stock

    # Compute embeddings for all headlines in the group
    emb_tensor = get_sentiment_features(headlines, finbert_model, finbert_tokenizer)

    # Average the embeddings across headlines: (768,)
    avg_embedding = emb_tensor.mean(dim=0)

    # Format date as YYYY-MM-DD and create file name
    date_str = date.strftime("%Y-%m-%d")
    file_name = f"finbert_emb_{date_str}_{stock_name}.pt"
    file_path = os.path.join(output_dir, file_name)

    # Save the averaged embedding
    torch.save(avg_embedding, file_path)

    # Store the metadata
    file_paths.append({'Date': date, 'stock_name': stock_name, 'file_path': file_path})

# Create the result dataframe
result_df = pd.DataFrame(file_paths)
result_df.to_csv("/content/drive/MyDrive/financial_data/finbert_emb_v2.csv")

Processing embeddings: 100%|██████████| 26593/26593 [1:46:52<00:00,  4.15it/s]


### Defining the Easy / Hard Sequence

In [ ]:
import pandas as pd
import os
from pandas.errors import EmptyDataError

input_dir = "/content/drive/MyDrive/financial_data/detect_easy"
all_data = []

for file in os.listdir(input_dir):
    if file.endswith('v2.csv'):
        file_path = os.path.join(input_dir, file)
        print(f"Checking file: {file_path}")

        try:
            df_stock = pd.read_csv(file_path)

            # Skip if DataFrame is empty after loading
            if df_stock.empty:
                print(f"Skipped empty DataFrame from file: {file_path}")
                continue

            all_data.append(df_stock)

        except EmptyDataError:
            print(f"Skipped empty file (no data): {file_path}")
            continue
        except pd.errors.ParserError as e:
            print(f"Skipped file due to parsing error: {file_path}, error: {e}")
            continue

if all_data:
    df = pd.concat(all_data, ignore_index=True)
    df['date'] = pd.to_datetime(df['date'])

    start_date = '2010-01-01'
    end_date = '2020-01-01'

    filtered_df = df[(df['date'] >= start_date) & (df['date'] < end_date)]
    print(filtered_df.head())
else:
    print("No valid data loaded.")

Checking file: /content/drive/MyDrive/financial_data/detect_easy/PLTR_metrics_v2.csv
Skipped empty file (no data): /content/drive/MyDrive/financial_data/detect_easy/PLTR_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/TSLA_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/KKR_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/AMD_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/NVDA_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/BX_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/BA_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/AMAT_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/LRCX_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect_easy/C_metrics_v2.csv
Checking file: /content/drive/MyDrive/financial_data/detect

In [ ]:
# Select the used model
filtered_df = filtered_df[filtered_df['model'].isin(['Maple728/TimeMoE-200M', 'amazon/chronos-bolt-base'])]
filtered_df

,stock,model,sequence_size,date,horizon,mse,mae,mape,r2,direction_correct
33935,TSLA,Maple728/TimeMoE-200M,20,2010-07-28,1,0.087182,0.295266,41.180549,0.643293,0
33936,TSLA,Maple728/TimeMoE-200M,20,2010-07-29,1,0.144703,0.380398,87.566487,0.643293,0
33937,TSLA,Maple728/TimeMoE-200M,20,2010-07-30,1,0.010705,0.103467,85.335369,0.643293,1
33938,TSLA,Maple728/TimeMoE-200M,20,2010-08-02,1,0.647776,0.804845,92.535308,0.643293,0
33939,TSLA,Maple728/TimeMoE-200M,20,2010-08-03,1,0.663092,0.814304,49.159096,0.643293,0
...,...,...,...,...,...,...,...,...,...,...
7846216,ADI,amazon/chronos-bolt-base,252,2019-12-24,5,0.001386,0.037234,2.516213,-3.277356,1
7846217,ADI,amazon/chronos-bolt-base,252,2019-12-26,5,0.046475,0.215580,14.557421,-3.277356,1
7846218,ADI,amazon/chronos-bolt-base,252,2019-12-27,5,0.003358,0.057946,3.997065,-3.277356,1
7846219,ADI,amazon/chronos-bolt-base,252,2019-12-30,5,0.023725,0.154031,11.129100,-3.277356,0


In [ ]:
import pandas as pd

# Assume filtered_df is your input DataFrame.
# Ensure the 'date' column is in datetime format.
filtered_df['date'] = pd.to_datetime(filtered_df['date'])

# Define the horizons.
short_horizons = [1, 5, 20, 126, 252]
long_horizons = [1, 5, 20, 126, 252]
ratio_mae = 0.25 # Group by [stock, horizon], take threshold value, < 0.35 proportion -> is_easy

# Initialize indicator columns
filtered_df['mae_indicator'] = None
filtered_df['direction_indicator'] = None

# ----------------------------
# (1) Compute short-term MAE indicator, for rows with horizon in short_horizons.
# ----------------------------
mask_short = filtered_df['horizon'].isin(short_horizons)
# Compute the 20th percentile threshold of MAE within each stock & horizon group.
filtered_df.loc[mask_short, 'mae_threshold'] = filtered_df.loc[mask_short].groupby(['sequence_size', 'stock', 'horizon'])['mae'].transform(lambda x: x.quantile(ratio_mae))
# Set mae_indicator to 1 if the row's MAE is below or equal to the threshold, else 0.
filtered_df.loc[mask_short, 'mae_indicator'] = (filtered_df.loc[mask_short, 'mae'] <= filtered_df.loc[mask_short, 'mae_threshold']).astype(int)

# ----------------------------
# (2) Compute long-term direction indicator, for rows with horizon in long_horizons.
# ----------------------------
mask_long = filtered_df['horizon'].isin(long_horizons)
filtered_df.loc[mask_long, 'direction_indicator'] = filtered_df.loc[mask_long, 'direction_correct'].astype(int)

# (Optional) Remove the temporary mae_threshold column if no longer needed.
filtered_df.drop(columns=['mae_threshold'], inplace=True)

filtered_df.head()

,stock,model,sequence_size,date,horizon,mse,mae,mape,r2,direction_correct,mae_indicator,direction_indicator
33935,TSLA,Maple728/TimeMoE-200M,20,2010-07-28,1,0.087182,0.295266,41.180549,0.643293,0,0,0
33936,TSLA,Maple728/TimeMoE-200M,20,2010-07-29,1,0.144703,0.380398,87.566487,0.643293,0,0,0
33937,TSLA,Maple728/TimeMoE-200M,20,2010-07-30,1,0.010705,0.103467,85.335369,0.643293,1,1,1
33938,TSLA,Maple728/TimeMoE-200M,20,2010-08-02,1,0.647776,0.804845,92.535308,0.643293,0,0,0
33939,TSLA,Maple728/TimeMoE-200M,20,2010-08-03,1,0.663092,0.814304,49.159096,0.643293,0,0,0


In [ ]:
# Assuming df is the input dataframe
df = filtered_df

# Filter the out horizons
df = df[df['horizon'].isin([5, 20])]

# Step 1: Preprocess the data
df['date'] = pd.to_datetime(df['date'])  # Ensure date is datetime

# Create a unique identifier for each model, sequence_size, horizon combination
df['combo_id'] = df['model'] + '_' + df['sequence_size'].astype(str) + '_horizon' + df['horizon'].astype(str)

# Step 2: Create indicator for direction_correct == 1 and mae < 0.25
df['indicator'] = ((df['direction_correct'] == 1)).astype(int) #  & (df['mae'] < 0.25)

# Step 3: Pivot to create columns for each combination
pivot_df = df.pivot_table(
    index=['date', 'stock'],
    columns='combo_id',
    values='indicator',
    fill_value=0  # Missing combinations are treated as failing the criteria
).reset_index()

# Step 4: Compute easy_indicator (1 if more than 50% of indicator columns are 1, else 0)
indicator_cols = [col for col in pivot_df.columns if col not in ['date', 'stock']]

# Calculate the fraction of columns with 1 in each row
fraction_ones = pivot_df[indicator_cols].mean(axis=1)

# Set easy_indicator to 1 if fraction > 0.5 else 0
pivot_df['easy_indicator'] = (fraction_ones >= 0.7).astype(int)

# Step 5: Ensure output dataframe has all required columns
result_df = pivot_df[['date', 'stock'] + indicator_cols + ['easy_indicator']]

# Optional: Sort by date and stock for clarity
result_df = result_df.sort_values(['date', 'stock'])

result_df = result_df[['date', 'stock', 'easy_indicator']]

result_df.head()

<ipython-input-30-2a8011717063>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])  # Ensure date is datetime
<ipython-input-30-2a8011717063>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['combo_id'] = df['model'] + '_' + df['sequence_size'].astype(str) + '_horizon' + df['horizon'].astype(str)
<ipython-input-30-2a8011717063>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = va

combo_id,date,stock,easy_indicator
0,2010-01-04,AAPL,0
1,2010-01-04,ACN,0
2,2010-01-04,ADBE,0
3,2010-01-04,ADI,0
4,2010-01-04,AMAT,0


In [ ]:
result_df['easy_indicator'].value_counts()

,count
easy_indicator,
0,94956
1,20823


In [ ]:
result_df.to_csv("/content/drive/MyDrive/financial_data/is_easy_5_10_v2.csv")

In [ ]:
import pandas as pd

# Assume filtered_df already contains the computed indicators.
# For example, filtered_df looks like:
#     stock       date  horizon   ...  mae_indicator  direction_indicator
# 320  TSLA 2011-06-28        1   ...              0                    0
# 321  TSLA 2011-06-29        1   ...              1                    1
# 322  TSLA 2011-06-30        1   ...              0                    0
# 323  TSLA 2011-07-01        1   ...              1                    1
#
# We'll now pivot to get one row per (stock, date) with one column per horizon indicator.

short_horizons = [5, 20]
long_horizons = [5, 20]

# Select short-term rows and pivot.
short_df = filtered_df[filtered_df['horizon'].isin(short_horizons)].copy()
short_pivot = short_df.pivot_table(index=['stock', 'date'],
                                   columns='horizon',
                                   values='mae_indicator',
                                   aggfunc='first').reset_index()

# Rename the pivoted columns to something like mae_indicator_1, mae_indicator_5, ...
short_pivot.columns = ['stock', 'date'] + [f"mae_indicator_{int(col)}" for col in short_pivot.columns[2:]]

# Select long-term rows and pivot.
long_df = filtered_df[filtered_df['horizon'].isin(long_horizons)].copy()
long_pivot = long_df.pivot_table(index=['stock','date'],
                                 columns='horizon',
                                 values='direction_indicator',
                                 aggfunc='first').reset_index()

# Rename the long-term columns.
long_pivot.columns = ['stock', 'date'] + [f"direction_indicator_{int(col)}" for col in long_pivot.columns[2:]]

# Merge the two pivoted DataFrames on stock and date.
merged_df = pd.merge(short_pivot, long_pivot, on=['stock', 'date'], how='outer')

# Fill missing values with 0 (i.e. if an indicator is missing, treat it as not easy).
indicator_cols = [f"mae_indicator_{h}" for h in short_horizons] + [f"direction_indicator_{h}" for h in long_horizons]
merged_df[indicator_cols] = merged_df[indicator_cols].fillna(0).astype(int)

# Compute overall "is_easy": It must be 1 for all relevant horizons.
merged_df['is_easy'] = merged_df[indicator_cols].min(axis=1).astype(int)

# Optionally, retain only the columns of interest.
result_df = merged_df[['stock', 'date'] + indicator_cols + ['is_easy']]

result_df['is_easy'].value_counts()

<ipython-input-9-3f232ad80348>:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df[indicator_cols] = merged_df[indicator_cols].fillna(0).astype(int)


,count
is_easy,
0,106929
1,8850


In [ ]:
merged_df

,stock,date,mae_indicator_5,mae_indicator_20,direction_indicator_5,direction_indicator_20,is_easy
0,AAPL,2010-01-04,0,0,0,1,0
1,AAPL,2010-01-05,0,0,0,1,0
2,AAPL,2010-01-06,0,0,0,1,0
3,AAPL,2010-01-07,0,0,1,1,0
4,AAPL,2010-01-08,0,0,0,1,0
...,...,...,...,...,...,...,...
115774,WFC,2019-12-24,1,1,1,1,1
115775,WFC,2019-12-26,0,1,1,0,0
115776,WFC,2019-12-27,1,1,1,1,1
115777,WFC,2019-12-30,1,1,1,1,1
